# 3) Exclamation Points: Rules vs. Reality

**Goal:** Count exclamation points per 10k words and compare.

# Setup: Load Texts

This notebook needs **Alice in Wonderland** and **Through the Looking-Glass** as input texts.

**How to provide the texts:**
1. Download books from Project Gutenberg (IDs 11 and 12) as txts. [go to https://www.gutenberg.org/ebooks/11 and https://www.gutenberg.org/ebooks/12]

2. Place two text files in the "data" folder with names:
   - `Wondeland.txt`  (Alice's Adventures in Wonderland)
   - `Looking-Glass.txt` (Through the Looking-Glass)

In [ ]:
import re
from pathlib import Path

In [11]:
def load_texts(local_crime: str = '../data/Crime-punishment.txt',
               local_brothers: str = '../data/The-BrothersKaramazov.txt'):
    """Load Crime and Punishment and The Brothers Karamazov texts from disk.

    Parameters
    ----------
    local_crime : str
        Path to Crime and Punishment text file. Defaults to '../data/Crime-punishment.txt'.
    local_brothers : str
        Path to The Brothers Karamazov text file. Defaults to '../data/The-BrothersKaramazov.txt'.

    Returns
    -------
    tuple[str, str]
        (crime_text, brothers_text).

    Raises
    ------
    FileNotFoundError
        If either file is missing.

    Extra Notes
    -----------
    - Using UTF-8 with `errors='ignore'` avoids codec exceptions on
      older Project Gutenberg dumps or inconsistent encodings.
    """
    p1, p2 = Path(local_crime), Path(local_brothers)

    # Fail fast with a clear message if a file is missing
    if not p1.exists():
        raise FileNotFoundError(
            f"Missing file: {p1}\n"
            "→ Please place 'Crime-punishment.txt' at this path or update load_texts(...)."
        )
    if not p2.exists():
        raise FileNotFoundError(
            f"Missing file: {p2}\n"
            "→ Please place 'The-BrothersKaramazov.txt' at this path or update load_texts(...)."
        )

    # Read the files (UTF-8; ignore undecodable bytes to stay robust)
    crime = p1.read_text(encoding='utf-8', errors='ignore')
    brothers = p2.read_text(encoding='utf-8', errors='ignore')
    return crime, brothers

def normalize(text: str) -> str:
    """Normalize a Gutenberg-like text for tokenization.

    Steps
    -----
    1) Heuristically strip Project Gutenberg headers/footers if present
       (looks for *** START ... *** END markers).
    2) Normalize newlines to '\n'.

    Parameters
    ----------
    text : str
        Raw text as loaded from disk (can be empty).

    Returns
    -------
    str
        Cleaned text suitable for tokenization and counting.
    """
    if not text:
        return ''
    # Clip to the main body if markers are present.
    start = text.find('*** START')
    end   = text.find('*** END')
    if start != -1 and end != -1 and end > start:
        text = text[start:end]
    # Normalize Windows line endings.
    return text.replace('\r\n', '\n')

# Load raw texts (forgiving: returns '' if a file is missing)
crime_raw, brothers_raw = load_texts()

# Normalize for tokenization
crime = normalize(crime_raw)
brothers = normalize(brothers_raw)

print(f"Crime and Punishment chars: {len(crime):,} | The Brothers Karamazov chars: {len(brothers):,}")


Crime and Punishment chars: 1,224,432 | The Brothers Karamazov chars: 1,956,247


In [12]:
WORD_RE = re.compile(r"[A-Za-z']+")  # keep apostrophes in words (e.g., don't -> don't)

def words(text: str):
    """Simple word tokenizer (lowercased, ASCII letters + apostrophes).

    Pros
    ----
    - Very fast and dependency-free.
    - Good enough for frequency/keyness demonstrations.

    Cons
    ----
    - No punctuation words, no sentence boundaries, no POS tags.
    - May treat possessives inconsistently across sources.

    Returns
    -------
    list[str]
        Lowercased word words.
    """
    return WORD_RE.findall(text.lower())


def sentences(text: str):
    """Naive sentence splitter using punctuation boundaries.

    Uses a regex to split on '.', '!', '?' followed by whitespace.
    Because this is heuristic, treat results as approximate.

    Returns
    -------
    list[str]
        Sentence-like strings.
    """
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]


# Tokenize and split sentences for both texts
crime_words = words(crime)
brothers_words = words(brothers)

crime_sentences = sentences(crime)
brothers_sentences = sentences(brothers)

print(f"Crime and Punishment words: {len(crime_words):,} | The Brothers Karamazov words: {len(brothers_words):,}")
print(f"Crime and Punishment sentences: {len(crime_sentences):,} | The Brothers Karamazov sentences: {len(brothers_sentences):,}")


Crime and Punishment words: 214,498 | The Brothers Karamazov words: 359,146
Crime and Punishment sentences: 16,994 | The Brothers Karamazov sentences: 19,234


### Count and Normalize

In [15]:
def exclamations_per_10k(text, tokens):
    count = text.count('!')
    per_10k = (count / max(1, len(tokens))) * 10000
    return count, per_10k


c_count, c_rate = exclamations_per_10k(crime, words(crime))
b_count, b_rate = exclamations_per_10k(brothers, words(brothers))

print(f"Crime and Punishment: {c_count} total | {c_rate:.2f} per 10k words")
print(f"The Brothers Karamazov: {b_count} total | {b_rate:.2f} per 10k words")


Crime and Punishment: 2376 total | 110.77 per 10k words
The Brothers Karamazov: 2734 total | 76.13 per 10k words


**Question:** Sample passages with many exclamation points. How do they shape voice, pacing, or mood?

In [16]:
# A) Sentence-level hotspots
def top_exclaim_sentences(sents, top_n=8, min_len=20):
    scored = [(s.count('!'), len(s), s) for s in sents if len(s) >= min_len]
    scored.sort(key=lambda x: (x[0], -x[1]), reverse=True)  # more !, then longer
    return [(cnt, s) for cnt, _, s in scored[:top_n] if cnt > 0]

# B) Clusters over sliding windows (tempo spikes)
def exclaim_clusters(sents, window=6, min_total=3, top_k=5):
    out = []
    for i in range(max(0, len(sents)-window+1)):
        chunk = " ".join(sents[i:i+window])
        c = chunk.count('!')
        if c >= min_total:
            out.append((c, i, " ".join(sents[i:i+window])))
    out.sort(reverse=True, key=lambda x: x[0])
    return out[:top_k]

def preview(s, n=300):
    return s if len(s) <= n else s[:n].rstrip() + " …"


print("=== Crime and Punishment: top sentences with ! ===")
for cnt, s in top_exclaim_sentences(crime_sentences):
    print(f"[! x{cnt}] {preview(s)}\n")

print("=== The Brothers Karamazov: top sentences with ! ===")
for cnt, s in top_exclaim_sentences(brothers_sentences):
    print(f"[! x{cnt}] {preview(s)}\n")

print("=== Crime and Punishment: exclamation clusters ===")
for c, i, chunk in exclaim_clusters(crime_sentences, window=6, min_total=3):
    print(f"[cluster ! x{c} | sentences {i}-{i+5}] {preview(chunk)}\n")

print("=== The Brothers Karamazov: exclamation clusters ===")
for c, i, chunk in exclaim_clusters(brothers_sentences, window=6, min_total=3):
    print(f"[cluster ! x{c} | sentences {i}-{i+5}] {preview(chunk)}\n")


=== Crime and Punishment: top sentences with ! ===
[! x4] She has not had a gallop in her
for the last ten years!”
    “She’ll jog along!”
    “Don’t you mind her, mates, bring a whip each of
you, get ready!”
    “All right!

[! x3] It’s not your
own tale you are telling!’ You must admit it’s a comi-
cal business!”
   “He-he-he!

[! x3] “That’s his notion!”
    “Talked himself silly!”
    “A fine clerk he is!”
    And so on, and so on.

[! x3] ”
    “Well, then, drop her!”
    “But I can’t drop her like that!”
    “Why can’t you?”
    “Well, I can’t, that’s all about it!

[! x3] “No!” he said, apparently abandoning all attempt
to keep up appearances with Porfiry, “it’s not worth it,
I don’t care about lessening the sentence!”
    “That’s just what I was afraid of!” Porfiry cried
warmly and, as it seemed, involuntarily.

[! x2] He-he!”
    “No, no!

[! x2] (Cough, cough, cough, cough!)
Again!

[! x2] Good-bye!”
    “He calls her Pashenka!

=== The Brothers Karamazov: top sentences with 